# AROM Assessment - Real-Time Pose Estimation

This notebook provides real-time AROM assessment using OpenCV camera and MediaPipe Pose.

## Structure
1. Core Assessment Logic (Constants, Helpers, Assessment Classes)
2. Real-Time Camera Integration (OpenCV + MediaPipe)
3. Assessment Service (Unified Interface)


In [ ]:
import math
import numpy as np
from typing import Dict, Optional, Tuple, List
from dataclasses import dataclass
from enum import Enum

# Replicate Flutter Offset as a simple class
class Offset:
    def __init__(self, x: float, y: float):
        self.dx = x
        self.dy = y
    
    def __sub__(self, other):
        return Offset(self.dx - other.dx, self.dy - other.dy)
    
    @property
    def distance(self):
        return math.sqrt(self.dx**2 + self.dy**2)
    
    def __repr__(self):
        return f"Offset({self.dx:.2f}, {self.dy:.2f})"


## Assessment Constants

These match `assessment_constants.dart` exactly.


In [ ]:
class AssessmentConstants:
    # Calf dorsiflexion thresholds
    calf_severe_threshold = 0.15  # Normalized displacement < 0.15 -> Severe
    calf_moderate_threshold = 0.30  # 0.15 <= displacement < 0.30 -> Moderate
    
    # Hamstring ROM thresholds
    hamstring_severe_threshold = 60.0  # Angle < 60° -> Severe
    hamstring_moderate_threshold = 80.0  # 60° <= Angle < 80° -> Moderate
    
    # Pelvic compensation threshold
    pelvic_compensation_threshold_norm = 0.05  # Vertical difference > 5% of body height proxy -> Warning
    
    # Triceps ROM thresholds
    triceps_severe_threshold = 90.0  # Angle < 90° -> Severe
    triceps_moderate_threshold = 135.0  # 90° <= Angle < 135° -> Moderate
    
    # Shoulder ROM thresholds
    shoulder_severe_threshold = 90.0  # Angle < 90° -> Severe
    shoulder_moderate_threshold = 110.0  # 90° <= Angle <= 110° -> Moderate
    shoulder_low_threshold = 150.0  # 111° <= Angle <= 150° -> Low pain
    
    # Chest ROM thresholds
    chest_severe_threshold = 45.0  # Angle < 45° -> Severe limitation
    chest_moderate_threshold = 90.0  # 45° <= Angle < 90° -> Moderate limitation
    
    # Biceps ROM thresholds
    biceps_severe_threshold = 150.0  # Angle > 150° -> Severe (Extended)
    biceps_moderate_threshold = 90.0  # 90° < Angle <= 150° -> Moderate
    
    # Quadriceps ROM thresholds
    quadriceps_severe_threshold = 160.0  # Angle >= 160° -> Severe (Extended)
    quadriceps_moderate_threshold = 100.0  # 100° <= Angle < 160° -> Moderate
    
    # Gluteal ROM thresholds
    gluteal_severe_threshold = 160.0  # Angle >= 160° -> Severe (Extended)
    gluteal_moderate_threshold = 100.0  # 100° <= Angle < 160° -> Moderate
    
    # Enhanced Hamstring ROM thresholds (for hip-knee-ankle assessment)
    hamstring_enhanced_severe_threshold = 160.0  # Angle >= 160° -> Severe (Extended)
    hamstring_enhanced_moderate_threshold = 100.0  # 100° <= Angle < 160° -> Moderate
    
    # Trunk ROM thresholds (for unified trunk muscle assessment)
    trunk_severe_threshold = 160.0  # Angle >= 160° -> Severe (Extended/Upright)
    trunk_moderate_threshold = 60.0  # 60° <= Angle < 160° -> Moderate

print("Assessment constants loaded")


## Pain Scale Mapping

Maps ROM levels to pain scores (0-10) and categorical levels (Low/Moderate/Severe).


In [ ]:
class PainScaleMapping:
    @staticmethod
    def map_to_pain_scale(rom_level: str) -> int:
        """Map ROM level to pain scale (0-10)"""
        mapping = {
            'severe': 9,  # 8-10: Severe limitation/pain
            'moderate': 6,  # 5-7: Moderate limitation/pain
            'low': 3,  # 2-4: Low limitation/pain
            'good': 1,  # 0-1: Good ROM/no pain
        }
        return mapping.get(rom_level, 5)  # Default moderate
    
    @staticmethod
    def map_to_categorical_pain_level(rom_level: str) -> str:
        """Map ROM level to categorical pain level"""
        mapping = {
            'severe': 'Severe',
            'moderate': 'Moderate',
            'low': 'Low',
            'good': 'Low',
        }
        return mapping.get(rom_level, 'Moderate')
    
    @staticmethod
    def get_score_color(score: int) -> str:
        """Get color based on pain score"""
        if score <= 3:
            return 'green'  # Good (0-3)
        elif score <= 7:
            return 'orange'  # Moderate (4-7)
        else:
            return 'red'  # Severe (8-10)

# Test pain scale mapping
print("Pain Scale Mapping Test:")
for rom in ['severe', 'moderate', 'low', 'good']:
    score = PainScaleMapping.map_to_pain_scale(rom)
    category = PainScaleMapping.map_to_categorical_pain_level(rom)
    color = PainScaleMapping.get_score_color(score)
    print(f"  {rom:8} -> Score: {score}, Category: {category:8}, Color: {color}")


## Helper Functions

Core angle calculation and utility functions used across all assessments.


In [ ]:
def calculate_angle_between_points(point_a: Offset, vertex: Offset, point_b: Offset) -> float:
    """
    Calculate angle between three points (vertex is middle point).
    Replicates _calculateAngleBetweenPoints from Flutter code.
    
    Returns angle in degrees (0-180).
    """
    v1 = point_a - vertex
    v2 = point_b - vertex
    
    dot = v1.dx * v2.dx + v1.dy * v2.dy
    mag1 = v1.distance
    mag2 = v2.distance
    
    if mag1 == 0 or mag2 == 0:
        return 0.0
    
    cos_theta = max(-1.0, min(1.0, dot / (mag1 * mag2)))
    radians = math.acos(cos_theta)
    return radians * 180.0 / math.pi

def calculate_vertical_angle(point1: Offset, point2: Offset) -> float:
    """
    Calculate vertical angle between two points.
    Replicates _calculateVerticalAngle from HamstringsAssessment.
    
    Returns angle in degrees (0-180).
    """
    vector = point2 - point1
    vertical_vector = Offset(0, -1)  # Vertical vector pointing upwards
    
    norm_vector = vector.distance
    norm_vertical = vertical_vector.distance
    
    if norm_vector == 0 or norm_vertical == 0:
        return 0.0
    
    cosine_angle = (vector.dx * vertical_vector.dx + vector.dy * vertical_vector.dy) / (norm_vector * norm_vertical)
    clamped_cosine = max(-1.0, min(1.0, cosine_angle))
    
    angle_radians = math.acos(clamped_cosine)
    angle_degrees = angle_radians * 180 / math.pi
    
    return angle_degrees

# Test angle calculations
print("Angle Calculation Test:")
p1 = Offset(0, 0)
p2 = Offset(1, 0)
p3 = Offset(0, 1)
angle = calculate_angle_between_points(p1, p2, p3)
print(f"  Right angle test: {angle:.2f}° (expected ~90°)")


## Assessment Result Data Class

Represents the result of an assessment, matching `AssessmentResult` from Flutter.


In [ ]:
@dataclass
class AssessmentResult:
    """Assessment result matching Flutter AssessmentResult class"""
    rom_level: str
    pain_score: int
    categorical_pain_level: str
    display_label: str
    display_color: str
    clinical_context: str
    additional_data: Dict = None
    alignment: Optional[str] = None
    compensation: Optional[str] = None
    
    def __post_init__(self):
        if self.additional_data is None:
            self.additional_data = {}
    
    @staticmethod
    def not_visible(muscle_group: str):
        """Create result when assessment cannot be performed"""
        return AssessmentResult(
            rom_level='unknown',
            pain_score=5,
            categorical_pain_level='Moderate',
            display_label=f'{muscle_group}: Not visible',
            display_color='white',
            clinical_context='Moderate'
        )
    
    @staticmethod
    def error(muscle_group: str):
        """Create result when assessment encounters an error"""
        return AssessmentResult(
            rom_level='error',
            pain_score=5,
            categorical_pain_level='Moderate',
            display_label=f'{muscle_group}: Error',
            display_color='red',
            clinical_context='Moderate'
        )
    
    @staticmethod
    def adjust_position(muscle_group: str):
        """Create result when position needs adjustment"""
        return AssessmentResult(
            rom_level='adjust',
            pain_score=5,
            categorical_pain_level='Moderate',
            display_label=f'{muscle_group}: Adjust position',
            display_color='yellow',
            clinical_context='Moderate'
        )
    
    def __repr__(self):
        return f"AssessmentResult(rom={self.rom_level}, pain={self.pain_score}, label='{self.display_label}')"


## Individual Muscle Assessments

Each assessment module replicates the corresponding Flutter class.


### 1. Triceps Assessment

**Landmarks Used:** `{side}Shoulder`, `{side}Elbow`, `{side}Wrist`  
**Angle Calculation:** Shoulder-Elbow-Wrist  
**Logic:** Lower angle = more flexed = less pain. Higher angle = more extended = more pain.


In [ ]:
class TricepsAssessment:
    @staticmethod
    def calculate_angle(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate triceps angle between shoulder, elbow, and wrist"""
        side_lower = side.lower()
        shoulder = landmarks.get(f'{side_lower}Shoulder')
        elbow = landmarks.get(f'{side_lower}Elbow')
        wrist = landmarks.get(f'{side_lower}Wrist')
        
        if shoulder is None or elbow is None or wrist is None:
            return None
        
        return calculate_angle_between_points(shoulder, elbow, wrist)
    
    @staticmethod
    def assess(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform triceps ROM assessment"""
        angle = TricepsAssessment.calculate_angle(landmarks, side)
        
        if angle is None:
            return AssessmentResult.not_visible('Triceps')
        
        # Evaluate ROM level
        if angle < AssessmentConstants.triceps_severe_threshold:
            rom_level = 'good'  # Angle < 90° -> Good (flexed)
        elif angle < AssessmentConstants.triceps_moderate_threshold:
            rom_level = 'moderate'  # 90° <= Angle < 135° -> Moderate
        else:
            rom_level = 'severe'  # Angle >= 135° -> Severe (extended)
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        
        # Get display label
        if rom_level == 'severe':
            display_label = f'Triceps ROM: Severe (<90°)'
        elif rom_level == 'moderate':
            display_label = f'Triceps ROM: Moderate (90-134°)'
        else:
            display_label = f'Triceps ROM: Good (>=135°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'side': side}
        )

print("TricepsAssessment loaded")


### 2. Shoulders Assessment

**Landmarks Used:** `{side}Hip`, `{side}Shoulder`, `{side}Elbow`  
**Angle Calculation:** Hip-Shoulder-Elbow  
**Logic:** Lower angle = arm raised higher = more pain. Higher angle = arm down = less pain.


In [ ]:
class ShouldersAssessment:
    @staticmethod
    def calculate_angle(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate shoulder angle between hip, shoulder, and elbow"""
        side_lower = side.lower()
        hip = landmarks.get(f'{side_lower}Hip')
        shoulder = landmarks.get(f'{side_lower}Shoulder')
        elbow = landmarks.get(f'{side_lower}Elbow')
        
        if hip is None or shoulder is None or elbow is None:
            return None
        
        return calculate_angle_between_points(hip, shoulder, elbow)
    
    @staticmethod
    def assess(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform shoulders ROM assessment"""
        angle = ShouldersAssessment.calculate_angle(landmarks, side)
        
        if angle is None:
            return AssessmentResult.not_visible('Shoulders')
        
        # Evaluate ROM level
        if angle < AssessmentConstants.shoulder_severe_threshold:
            rom_level = 'severe'  # Angle < 90° -> Severe Pain
        elif angle <= AssessmentConstants.shoulder_moderate_threshold:
            rom_level = 'moderate'  # 90° <= Angle <= 110° -> Moderate Pain
        elif angle <= AssessmentConstants.shoulder_low_threshold:
            rom_level = 'low'  # 111° <= Angle <= 150° -> Low Pain
        else:
            rom_level = 'good'  # Angle > 150° -> Good Mobility/Low Pain
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        
        # Get display label
        if rom_level == 'severe':
            display_label = 'Shoulder Pain: Severe (<90°)'
        elif rom_level == 'moderate':
            display_label = 'Shoulder Pain: Moderate (90-110°)'
        elif rom_level == 'low':
            display_label = 'Shoulder Pain: Low (111-150°)'
        else:
            display_label = 'Shoulder Mobility: Good (>=151°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'side': side}
        )

print("ShouldersAssessment loaded")


### 3. Biceps Assessment

**Landmarks Used:** `{side}Shoulder`, `{side}Elbow`, `{side}Wrist`  
**Angle Calculation:** Shoulder-Elbow-Wrist  
**Logic:** Higher angle = more extended = more pain. Lower angle = more flexed = less pain.


In [ ]:
class BicepsAssessment:
    @staticmethod
    def calculate_angle(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate biceps angle between shoulder, elbow, and wrist"""
        side_lower = side.lower()
        shoulder = landmarks.get(f'{side_lower}Shoulder')
        elbow = landmarks.get(f'{side_lower}Elbow')
        wrist = landmarks.get(f'{side_lower}Wrist')
        
        if shoulder is None or elbow is None or wrist is None:
            return None
        
        return calculate_angle_between_points(shoulder, elbow, wrist)
    
    @staticmethod
    def assess(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform biceps ROM assessment"""
        angle = BicepsAssessment.calculate_angle(landmarks, side)
        
        if angle is None:
            return AssessmentResult.not_visible('Biceps')
        
        # Evaluate ROM level
        if angle > AssessmentConstants.biceps_severe_threshold:
            rom_level = 'severe'  # Angle > 150° -> Severe (Extended)
        elif angle > AssessmentConstants.biceps_moderate_threshold:
            rom_level = 'moderate'  # 90° < Angle <= 150° -> Moderate
        else:
            rom_level = 'low'  # Angle <= 90° -> Low pain (Flexed)
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        
        # Get display label
        if rom_level == 'severe':
            display_label = f'Biceps ROM: Severe (> {int(AssessmentConstants.biceps_severe_threshold)}°)'
        elif rom_level == 'moderate':
            display_label = f'Biceps ROM: Moderate ({int(AssessmentConstants.biceps_moderate_threshold)}-{int(AssessmentConstants.biceps_severe_threshold)}°)'
        else:
            display_label = f'Biceps ROM: Low (< {int(AssessmentConstants.biceps_moderate_threshold)}°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'side': side}
        )

print("BicepsAssessment loaded")


### 4. Chest Assessment

**Landmarks Used:** `{side}Hip`, `{side}Shoulder`, `{side}Wrist`  
**Angle Calculation:** Hip-Shoulder-Wrist  
**Logic:** Higher angle = better forward elevation = less pain.


In [ ]:
class ChestAssessment:
    @staticmethod
    def calculate_angle(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate chest forward elevation angle between hip, shoulder, and wrist"""
        side_lower = side.lower()
        hip = landmarks.get(f'{side_lower}Hip')
        shoulder = landmarks.get(f'{side_lower}Shoulder')
        wrist = landmarks.get(f'{side_lower}Wrist')
        
        if hip is None or shoulder is None or wrist is None:
            return None
        
        return calculate_angle_between_points(hip, shoulder, wrist)
    
    @staticmethod
    def assess(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform chest ROM assessment"""
        angle = ChestAssessment.calculate_angle(landmarks, side)
        
        if angle is None:
            return AssessmentResult.not_visible('Chest')
        
        # Evaluate ROM level
        if angle < AssessmentConstants.chest_severe_threshold:
            rom_level = 'severe'  # Angle < 45° -> Severe
        elif angle < AssessmentConstants.chest_moderate_threshold:
            rom_level = 'moderate'  # 45° <= Angle < 90° -> Moderate
        else:
            rom_level = 'good'  # Angle >= 90° -> Good
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        
        # Get display label
        if rom_level == 'severe':
            display_label = 'Chest ROM: Severe (< 45°)'
        elif rom_level == 'moderate':
            display_label = 'Chest ROM: Moderate (45-90°)'
        else:
            display_label = 'Chest ROM: Good (≥ 90°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'side': side}
        )

print("ChestAssessment loaded")


### 5. Quadriceps Assessment

**Landmarks Used:** `{side}Hip`, `{side}Knee`, `{side}Ankle`  
**Angle Calculation:** Hip-Knee-Ankle  
**Logic:** Higher angle = more extended = more pain. Lower angle = more flexed = less pain.


In [ ]:
class QuadricepsAssessment:
    @staticmethod
    def calculate_angle(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate quadriceps angle between hip, knee, and ankle"""
        side_lower = side.lower()
        hip = landmarks.get(f'{side_lower}Hip')
        knee = landmarks.get(f'{side_lower}Knee')
        ankle = landmarks.get(f'{side_lower}Ankle')
        
        if hip is None or knee is None or ankle is None:
            return None
        
        return calculate_angle_between_points(hip, knee, ankle)
    
    @staticmethod
    def assess(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform quadriceps ROM assessment"""
        angle = QuadricepsAssessment.calculate_angle(landmarks, side)
        
        if angle is None:
            return AssessmentResult.not_visible('Quadriceps')
        
        # Evaluate ROM level
        if angle >= AssessmentConstants.quadriceps_severe_threshold:
            rom_level = 'severe'  # Angle >= 160° -> Severe (Extended)
        elif angle >= AssessmentConstants.quadriceps_moderate_threshold:
            rom_level = 'moderate'  # 100° <= Angle < 160° -> Moderate
        else:
            rom_level = 'low'  # Angle < 100° -> Low pain (Flexed)
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        
        # Get display label
        if rom_level == 'severe':
            display_label = f'Quadriceps ROM: Severe (>= {int(AssessmentConstants.quadriceps_severe_threshold)}°)'
        elif rom_level == 'moderate':
            display_label = f'Quadriceps ROM: Moderate ({int(AssessmentConstants.quadriceps_moderate_threshold)}-{int(AssessmentConstants.quadriceps_severe_threshold) - 1}°)'
        else:
            display_label = f'Quadriceps ROM: Low (< {int(AssessmentConstants.quadriceps_moderate_threshold)}°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'side': side}
        )

print("QuadricepsAssessment loaded")


### 6. Hamstrings Assessment (Simple Version)

**Landmarks Used:** `{side}Hip`, `{side}Ankle`  
**Angle Calculation:** Vertical angle from hip to ankle  
**Logic:** Lower angle = less extension = more pain. Higher angle = more extension = less pain.


In [ ]:
class HamstringsAssessment:
    @staticmethod
    def calculate_angle(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate hamstring angle (angle between hip-ankle and vertical axis)"""
        side_lower = side.lower()
        hip = landmarks.get(f'{side_lower}Hip')
        ankle = landmarks.get(f'{side_lower}Ankle')
        
        if hip is None or ankle is None:
            return None
        
        return calculate_vertical_angle(hip, ankle)
    
    @staticmethod
    def check_pelvic_compensation(landmarks: Dict[str, Offset]) -> str:
        """Check for pelvic compensation"""
        hip_l = landmarks.get('leftHip')
        hip_r = landmarks.get('rightHip')
        shoulder_r = landmarks.get('rightShoulder')
        shoulder_l = landmarks.get('leftShoulder')
        
        if hip_l is None or hip_r is None or shoulder_r is None or shoulder_l is None:
            return "Compensation: N/A"
        
        # Check for pelvic compensation
        vertical_hip_difference = abs(hip_r.dy - hip_l.dy)
        avg_shoulder_y = (shoulder_r.dy + shoulder_l.dy) / 2
        avg_hip_y = (hip_r.dy + hip_l.dy) / 2
        torso_height_proxy = abs(avg_shoulder_y - avg_hip_y)
        
        if torso_height_proxy > 5:
            norm_vertical_hip_difference = vertical_hip_difference / torso_height_proxy
            
            if norm_vertical_hip_difference > AssessmentConstants.pelvic_compensation_threshold_norm:
                return f"Compensation: Pelvic Tilt ({norm_vertical_hip_difference:.2f})"
            else:
                return "Compensation: Stable"
        else:
            return "Compensation: Cannot assess (Torso too flat)"
    
    @staticmethod
    def assess(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform hamstrings ROM assessment"""
        angle = HamstringsAssessment.calculate_angle(landmarks, side)
        
        if angle is None:
            return AssessmentResult.not_visible('Hamstring')
        
        # Evaluate ROM level
        if angle < AssessmentConstants.hamstring_severe_threshold:
            rom_level = 'severe'  # Angle < 60° -> Severe
        elif angle < AssessmentConstants.hamstring_moderate_threshold:
            rom_level = 'moderate'  # 60° <= Angle < 80° -> Moderate
        else:
            rom_level = 'good'  # Angle >= 80° -> Good
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        compensation = HamstringsAssessment.check_pelvic_compensation(landmarks)
        
        # Get display label
        if rom_level == 'severe':
            display_label = f'Hamstring ROM: Severe (< {int(AssessmentConstants.hamstring_severe_threshold)}°)'
        elif rom_level == 'moderate':
            display_label = f'Hamstring ROM: Moderate ({int(AssessmentConstants.hamstring_severe_threshold)}-{int(AssessmentConstants.hamstring_moderate_threshold)}°)'
        else:
            display_label = f'Hamstring ROM: Good (> {int(AssessmentConstants.hamstring_moderate_threshold)}°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'side': side},
            compensation=compensation
        )

print("HamstringsAssessment loaded")


### 7. Gluteals Assessment (Enhanced)

**Landmarks Used:** `{side}Shoulder`, `{side}Hip`, `{side}Knee`  
**Angle Calculation:** Shoulder-Hip-Knee  
**Logic:** Higher angle = more extended = more pain. Lower angle = more flexed = less pain.


In [ ]:
class GluteHamAssessment:
    @staticmethod
    def calculate_gluteal_angle(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate gluteal angle between shoulder, hip, and knee"""
        side_lower = side.lower()
        shoulder = landmarks.get(f'{side_lower}Shoulder')
        hip = landmarks.get(f'{side_lower}Hip')
        knee = landmarks.get(f'{side_lower}Knee')
        
        if shoulder is None or hip is None or knee is None:
            return None
        
        return calculate_angle_between_points(shoulder, hip, knee)
    
    @staticmethod
    def calculate_hamstring_angle(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate hamstring angle between hip, knee, and ankle (enhanced version)"""
        side_lower = side.lower()
        hip = landmarks.get(f'{side_lower}Hip')
        knee = landmarks.get(f'{side_lower}Knee')
        ankle = landmarks.get(f'{side_lower}Ankle')
        
        if hip is None or knee is None or ankle is None:
            return None
        
        return calculate_angle_between_points(hip, knee, ankle)
    
    @staticmethod
    def assess_gluteals(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform gluteal ROM assessment"""
        angle = GluteHamAssessment.calculate_gluteal_angle(landmarks, side)
        
        if angle is None:
            return AssessmentResult.not_visible('Gluteals')
        
        # Evaluate ROM level
        if angle >= AssessmentConstants.gluteal_severe_threshold:
            rom_level = 'severe'  # Angle >= 160° -> Severe (Extended)
        elif angle >= AssessmentConstants.gluteal_moderate_threshold:
            rom_level = 'moderate'  # 100° <= Angle < 160° -> Moderate
        else:
            rom_level = 'low'  # Angle < 100° -> Low pain (Flexed)
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        
        # Get display label
        if rom_level == 'severe':
            display_label = f'Gluteal ROM: Severe (>= {int(AssessmentConstants.gluteal_severe_threshold)}°)'
        elif rom_level == 'moderate':
            display_label = f'Gluteal ROM: Moderate ({int(AssessmentConstants.gluteal_moderate_threshold)}-{int(AssessmentConstants.gluteal_severe_threshold) - 1}°)'
        else:
            display_label = f'Gluteal ROM: Low (< {int(AssessmentConstants.gluteal_moderate_threshold)}°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'side': side, 'muscleGroup': 'Gluteals'}
        )
    
    @staticmethod
    def assess_hamstrings(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform hamstring ROM assessment (enhanced version)"""
        angle = GluteHamAssessment.calculate_hamstring_angle(landmarks, side)
        
        if angle is None:
            return AssessmentResult.not_visible('Hamstrings')
        
        # Evaluate ROM level
        if angle >= AssessmentConstants.hamstring_enhanced_severe_threshold:
            rom_level = 'severe'  # Angle >= 160° -> Severe (Extended)
        elif angle >= AssessmentConstants.hamstring_enhanced_moderate_threshold:
            rom_level = 'moderate'  # 100° <= Angle < 160° -> Moderate
        else:
            rom_level = 'low'  # Angle < 100° -> Low pain (Flexed)
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        
        # Get display label
        if rom_level == 'severe':
            display_label = f'Hamstring ROM: Severe (>= {int(AssessmentConstants.hamstring_enhanced_severe_threshold)}°)'
        elif rom_level == 'moderate':
            display_label = f'Hamstring ROM: Moderate ({int(AssessmentConstants.hamstring_enhanced_moderate_threshold)}-{int(AssessmentConstants.hamstring_enhanced_severe_threshold) - 1}°)'
        else:
            display_label = f'Hamstring ROM: Low (< {int(AssessmentConstants.hamstring_enhanced_moderate_threshold)}°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'side': side, 'muscleGroup': 'Hamstrings'}
        )

print("GluteHamAssessment loaded")


### 8. Calves Assessment

**Landmarks Used:** `{side}Hip`, `{side}Knee`, `{side}Ankle`  
**Calculation:** Normalized horizontal displacement (knee-ankle) / body segment height  
**Logic:** Lower displacement = less dorsiflexion = more pain. Higher displacement = more dorsiflexion = less pain.


In [ ]:
class CalvesAssessment:
    @staticmethod
    def calculate_normalized_displacement(landmarks: Dict[str, Offset], side: str) -> Optional[float]:
        """Calculate normalized displacement for calf dorsiflexion"""
        side_lower = side.lower()
        hip = landmarks.get(f'{side_lower}Hip')
        knee = landmarks.get(f'{side_lower}Knee')
        ankle = landmarks.get(f'{side_lower}Ankle')
        
        if hip is None or knee is None or ankle is None:
            return None
        
        # Calculate horizontal displacement between knee and ankle
        horizontal_displacement = knee.dx - ankle.dx
        
        # Calculate vertical distance between hip and ankle as body height proxy
        body_segment_height = abs(hip.dy - ankle.dy)
        
        if body_segment_height < 10:
            return None  # Position adjustment needed
        
        # Normalize horizontal displacement by body segment height
        return horizontal_displacement / body_segment_height
    
    @staticmethod
    def check_alignment(landmarks: Dict[str, Offset], side: str) -> str:
        """Check knee-over-ankle alignment"""
        side_lower = side.lower()
        knee = landmarks.get(f'{side_lower}Knee')
        ankle = landmarks.get(f'{side_lower}Ankle')
        
        if knee is None or ankle is None:
            return "Alignment: N/A"
        
        if knee.dx > ankle.dx:
            return "Alignment: Knee Forward"
        else:
            return "Alignment: Knee Behind/Inline"
    
    @staticmethod
    def assess(landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform calves ROM assessment"""
        normalized_displacement = CalvesAssessment.calculate_normalized_displacement(landmarks, side)
        
        if normalized_displacement is None:
            # Check if it's a position issue or missing landmarks
            side_lower = side.lower()
            hip = landmarks.get(f'{side_lower}Hip')
            knee = landmarks.get(f'{side_lower}Knee')
            ankle = landmarks.get(f'{side_lower}Ankle')
            
            if hip is None or knee is None or ankle is None:
                return AssessmentResult.not_visible('Calf')
            else:
                return AssessmentResult.adjust_position('Calf')
        
        abs_norm_displacement = abs(normalized_displacement)
        
        # Evaluate ROM level
        if abs_norm_displacement < AssessmentConstants.calf_severe_threshold:
            rom_level = 'severe'  # < 0.15 -> Severe
        elif abs_norm_displacement < AssessmentConstants.calf_moderate_threshold:
            rom_level = 'moderate'  # 0.15-0.30 -> Moderate
        else:
            rom_level = 'good'  # >= 0.30 -> Good
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        alignment = CalvesAssessment.check_alignment(landmarks, side)
        
        # Get display label
        if rom_level == 'severe':
            display_label = f'Calf ROM: Severe (< {AssessmentConstants.calf_severe_threshold:.2f})'
        elif rom_level == 'moderate':
            display_label = f'Calf ROM: Moderate ({AssessmentConstants.calf_severe_threshold:.2f}-{AssessmentConstants.calf_moderate_threshold:.2f})'
        else:
            display_label = f'Calf ROM: Good (> {AssessmentConstants.calf_moderate_threshold:.2f})'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={
                'normalizedDisplacement': normalized_displacement,
                'absNormalizedDisplacement': abs_norm_displacement,
                'side': side
            },
            alignment=alignment
        )

print("CalvesAssessment loaded")


### 9. Trunk Assessment (Abdominals, Obliques, Lower Back, Multifidus)

**Landmarks Used:** `leftShoulder`, `rightShoulder`, `leftHip`, `rightHip`, `leftKnee`, `rightKnee`  
**Angle Calculation:** Midpoint Shoulder-Midpoint Hip-Midpoint Knee  
**Logic:** Higher angle = more upright/extended = more pain. Lower angle = more flexed = less pain.


In [ ]:
class BTrunkAssessment:
    @staticmethod
    def calculate_trunk_angle(landmarks: Dict[str, Offset]) -> Optional[float]:
        """Calculate trunk angle using shoulder-hip-knee landmarks"""
        left_shoulder = landmarks.get('leftShoulder')
        right_shoulder = landmarks.get('rightShoulder')
        left_hip = landmarks.get('leftHip')
        right_hip = landmarks.get('rightHip')
        left_knee = landmarks.get('leftKnee')
        right_knee = landmarks.get('rightKnee')
        
        if (left_shoulder is None or right_shoulder is None or 
            left_hip is None or right_hip is None or 
            left_knee is None or right_knee is None):
            return None
        
        # Calculate midpoint landmarks for trunk assessment
        shoulder_mid = Offset(
            (left_shoulder.dx + right_shoulder.dx) / 2,
            (left_shoulder.dy + right_shoulder.dy) / 2
        )
        hip_mid = Offset(
            (left_hip.dx + right_hip.dx) / 2,
            (left_hip.dy + right_hip.dy) / 2
        )
        knee_mid = Offset(
            (left_knee.dx + right_knee.dx) / 2,
            (left_knee.dy + right_knee.dy) / 2
        )
        
        # Calculate trunk angle: shoulder-hip-knee
        return calculate_angle_between_points(shoulder_mid, hip_mid, knee_mid)
    
    @staticmethod
    def assess(landmarks: Dict[str, Offset], muscle_type: str) -> AssessmentResult:
        """Perform unified trunk assessment for all four muscle groups"""
        angle = BTrunkAssessment.calculate_trunk_angle(landmarks)
        
        if angle is None:
            return AssessmentResult.not_visible(muscle_type)
        
        # Evaluate ROM level
        if angle >= AssessmentConstants.trunk_severe_threshold:
            rom_level = 'severe'  # Angle >= 160° -> Severe (Extended/Upright)
        elif angle >= AssessmentConstants.trunk_moderate_threshold:
            rom_level = 'moderate'  # 60° <= Angle < 160° -> Moderate
        else:
            rom_level = 'low'  # Angle < 60° -> Low pain (Flexed)
        
        pain_score = PainScaleMapping.map_to_pain_scale(rom_level)
        categorical_pain = PainScaleMapping.map_to_categorical_pain_level(rom_level)
        
        # Get display label
        if rom_level == 'severe':
            display_label = f'{muscle_type} ROM: Severe (>=160°)'
        elif rom_level == 'moderate':
            display_label = f'{muscle_type} ROM: Moderate (60-160°)'
        else:
            display_label = f'{muscle_type} ROM: Low (<60°)'
        
        return AssessmentResult(
            rom_level=rom_level,
            pain_score=pain_score,
            categorical_pain_level=categorical_pain,
            display_label=display_label,
            display_color=PainScaleMapping.get_score_color(pain_score),
            clinical_context=categorical_pain,
            additional_data={'angle': angle, 'muscleType': muscle_type, 'side': 'center'}
        )
    
    @staticmethod
    def assess_abdominals(landmarks: Dict[str, Offset]) -> AssessmentResult:
        return BTrunkAssessment.assess(landmarks, 'Abdominals')
    
    @staticmethod
    def assess_obliques(landmarks: Dict[str, Offset]) -> AssessmentResult:
        return BTrunkAssessment.assess(landmarks, 'Obliques')
    
    @staticmethod
    def assess_lower_back(landmarks: Dict[str, Offset]) -> AssessmentResult:
        return BTrunkAssessment.assess(landmarks, 'Lower Back')
    
    @staticmethod
    def assess_multifidus(landmarks: Dict[str, Offset]) -> AssessmentResult:
        return BTrunkAssessment.assess(landmarks, 'Multifidus')

print("BTrunkAssessment loaded")


## Unified Assessment Service

Replicates `AssessmentService` from Flutter - provides a unified interface for all assessments.


In [ ]:
class AssessmentService:
    """Unified assessment service that provides a consistent API for all muscle group assessments"""
    
    @staticmethod
    def assess(muscle_group: str, landmarks: Dict[str, Offset], side: str) -> AssessmentResult:
        """Perform assessment for the specified muscle group and side"""
        muscle_lower = muscle_group.lower()
        
        if muscle_lower == 'triceps':
            return TricepsAssessment.assess(landmarks, side)
        elif muscle_lower == 'shoulders':
            return ShouldersAssessment.assess(landmarks, side)
        elif muscle_lower == 'hamstrings':
            return GluteHamAssessment.assess_hamstrings(landmarks, side)
        elif muscle_lower == 'gluteals':
            return GluteHamAssessment.assess_gluteals(landmarks, side)
        elif muscle_lower in ['calf', 'calves']:
            return CalvesAssessment.assess(landmarks, side)
        elif muscle_lower == 'chest':
            return ChestAssessment.assess(landmarks, side)
        elif muscle_lower == 'biceps':
            return BicepsAssessment.assess(landmarks, side)
        elif muscle_lower == 'quadriceps':
            return QuadricepsAssessment.assess(landmarks, side)
        elif muscle_lower == 'abdominals':
            return BTrunkAssessment.assess_abdominals(landmarks)
        elif muscle_lower == 'obliques':
            return BTrunkAssessment.assess_obliques(landmarks)
        elif muscle_lower == 'lower back':
            return BTrunkAssessment.assess_lower_back(landmarks)
        elif muscle_lower == 'multifidus':
            return BTrunkAssessment.assess_multifidus(landmarks)
        else:
            return AssessmentResult.error(muscle_group)
    
    @staticmethod
    def get_available_muscle_groups() -> List[str]:
        """Get available muscle groups for assessment"""
        return ['Triceps', 'Shoulders', 'Hamstrings', 'Gluteals', 'Calf', 'Chest', 
                'Biceps', 'Quadriceps', 'Abdominals', 'Obliques', 'Lower Back', 'Multifidus']
    
    @staticmethod
    def get_available_sides() -> List[str]:
        """Get available sides for assessment"""
        return ['Left', 'Right']
    
    @staticmethod
    def is_supported_muscle_group(muscle_group: str) -> bool:
        """Check if a muscle group is supported"""
        available = AssessmentService.get_available_muscle_groups()
        return any(group.lower() == muscle_group.lower() for group in available)

print("AssessmentService loaded")


## Test Cases and Validation

Create sample landmark data and test all assessments to verify correctness.


## Real-Time Pose Estimation

The following sections provide real-time camera integration with MediaPipe Pose detection.


## MediaPipe Pose Detection Integration

MediaPipe Pose provides real-time pose detection compatible with the Flutter implementation.


In [ ]:
# Install required packages (uncomment to install):
# !pip install mediapipe opencv-python

try:
    import cv2
    import mediapipe as mp
    import numpy as np
    MEDIAPIPE_AVAILABLE = True
except ImportError:
    MEDIAPIPE_AVAILABLE = False
    print("MediaPipe or OpenCV not installed. Install with: pip install mediapipe opencv-python")

if MEDIAPIPE_AVAILABLE:
    class MediaPipePoseDetector:
        """
        MediaPipe Pose Detection wrapper that replicates Flutter's PoseDetectionService behavior.
        
        Key differences from Flutter:
        - Flutter uses Google ML Kit (mobile-only)
        - This uses MediaPipe (cross-platform Python alternative)
        - Both provide 33 body landmarks with normalized coordinates (0-1)
        """
        
        def __init__(self, 
                     model_complexity=1,
                     min_detection_confidence=0.5,
                     min_tracking_confidence=0.5,
                     is_front_camera=False):
            self.mp_pose = mp.solutions.pose
            self.pose = self.mp_pose.Pose(
                static_image_mode=False,
                model_complexity=model_complexity,
                enable_segmentation=False,
                min_detection_confidence=min_detection_confidence,
                min_tracking_confidence=min_tracking_confidence
            )
            self.mp_drawing = mp.solutions.drawing_utils
            self.mp_drawing_styles = mp.solutions.drawing_styles
            self.is_front_camera = is_front_camera
            self.last_image_size = None  # Track image size for coordinate normalization
        
        def detect_landmarks_from_image(self, image_path: str) -> Optional[Dict[str, Offset]]:
            """Detect pose landmarks from an image file and convert to normalized format (0-1)"""
            image = cv2.imread(image_path)
            if image is None:
                return None
            
            return self.detect_landmarks_from_frame(image)
        
        def detect_landmarks_from_frame(self, frame) -> Optional[Dict[str, Offset]]:
            """
            Detect pose landmarks from a camera frame and convert to normalized format (0-1).
            
            This replicates Flutter's getPoseLandmarks() behavior:
            - Normalizes coordinates to 0.0-1.0 range (matching Flutter)
            - Mirrors horizontally for front camera (matching Flutter)
            - Returns Map<String, Offset> with same landmark names as Flutter
            """
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.pose.process(image_rgb)
            
            if not results.pose_landmarks:
                return None
            
            # Store image size for coordinate normalization (matching Flutter's _lastImageSize)
            h, w = frame.shape[:2]
            self.last_image_size = (w, h)
            
            # Convert MediaPipe landmarks to normalized Offset format (0-1 range)
            # This matches Flutter's getPoseLandmarks() which normalizes ML Kit coordinates
            landmarks = {}
            mp_landmarks = results.pose_landmarks.landmark
            
            # Map MediaPipe landmarks to Flutter's naming convention
            # Flutter uses lowercase camelCase: 'leftShoulder', 'rightElbow', etc.
            landmark_mapping = {
                'leftShoulder': mp.solutions.pose.PoseLandmark.LEFT_SHOULDER,
                'rightShoulder': mp.solutions.pose.PoseLandmark.RIGHT_SHOULDER,
                'leftElbow': mp.solutions.pose.PoseLandmark.LEFT_ELBOW,
                'rightElbow': mp.solutions.pose.PoseLandmark.RIGHT_ELBOW,
                'leftWrist': mp.solutions.pose.PoseLandmark.LEFT_WRIST,
                'rightWrist': mp.solutions.pose.PoseLandmark.RIGHT_WRIST,
                'leftHip': mp.solutions.pose.PoseLandmark.LEFT_HIP,
                'rightHip': mp.solutions.pose.PoseLandmark.RIGHT_HIP,
                'leftKnee': mp.solutions.pose.PoseLandmark.LEFT_KNEE,
                'rightKnee': mp.solutions.pose.PoseLandmark.RIGHT_KNEE,
                'leftAnkle': mp.solutions.pose.PoseLandmark.LEFT_ANKLE,
                'rightAnkle': mp.solutions.pose.PoseLandmark.RIGHT_ANKLE,
            }
            
            # Helper to normalize and mirror (matching Flutter's _toNormalized function)
            def _to_normalized(x: float, y: float) -> Offset:
                # MediaPipe already provides normalized coordinates (0-1)
                # Clamp to valid range (matching Flutter's clamp)
                nx = max(0.0, min(1.0, x))
                ny = max(0.0, min(1.0, y))
                
                # Mirror horizontally for front camera (matching Flutter's behavior)
                if self.is_front_camera:
                    nx = 1.0 - nx
                
                return Offset(nx, ny)
            
            for name, mp_index in landmark_mapping.items():
                landmark = mp_landmarks[mp_index]
                # MediaPipe provides normalized coordinates (0-1), same as Flutter after normalization
                # Note: MediaPipe Y increases downward (same as Flutter)
                landmarks[name] = _to_normalized(landmark.x, landmark.y)
            
            return landmarks
        
        def draw_landmarks(self, frame, results):
            """Draw pose landmarks and connections on frame"""
            if results.pose_landmarks:
                self.mp_drawing.draw_landmarks(
                    frame,
                    results.pose_landmarks,
                    self.mp_pose.POSE_CONNECTIONS,
                    landmark_drawing_spec=self.mp_drawing_styles.get_default_pose_landmarks_style()
                )
            return frame
    
    print("MediaPipePoseDetector class loaded successfully (matches Flutter's coordinate system)")
else:
    print("MediaPipe not available - install with: pip install mediapipe opencv-python")


## Real-Time Camera Assessment with OpenCV and MediaPipe

This section provides a complete real-time camera implementation that:
1. Captures video from your webcam using OpenCV
2. Detects poses using MediaPipe Pose
3. Runs AROM assessments on detected poses
4. Displays results in real-time


In [ ]:
if MEDIAPIPE_AVAILABLE:
    import time
    
    class RealTimeAROMAssessment:
        """
        Real-time AROM assessment using OpenCV camera and MediaPipe Pose.
        
        This replicates the exact flow from Flutter's c_camera.dart:
        1. Start image stream (camera loop)
        2. For each frame:
           - Throttle processing (120ms minimum between frames)
           - Detect poses using pose detection service
           - Extract landmarks (normalized 0-1 coordinates)
           - Validate landmarks (empty check, minimum count)
           - Run assessment using AssessmentService
           - Update results
        """
        
        def __init__(self, 
                     muscle_group='Triceps',
                     side='Right',
                     camera_index=0,
                     show_skeleton=True,
                     show_assessment=True,
                     is_front_camera=False):
            self.muscle_group = muscle_group
            self.side = side
            self.camera_index = camera_index
            self.show_skeleton = show_skeleton
            self.show_assessment = show_assessment
            self.detector = MediaPipePoseDetector(is_front_camera=is_front_camera)
            self.cap = None
            self.current_result = None
            
            # Frame throttling state (matching Flutter's _throttleTimer and _lastProcessedTime)
            self.processing_frame = False
            self.last_processed_time = None  # milliseconds timestamp
            self.min_frame_interval_ms = 120  # Minimum 120ms between frames (matching Flutter)
            
            # Performance monitoring (matching Flutter's FPS tracking)
            self.frame_count = 0
            self.last_fps_time = 0
            self.current_fps = 0.0
        
        def start_camera(self):
            """Initialize camera"""
            self.cap = cv2.VideoCapture(self.camera_index)
            if not self.cap.isOpened():
                raise Exception(f"Could not open camera {self.camera_index}")
            
            # Set camera properties
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
            print(f"Camera {self.camera_index} initialized")
        
        def draw_assessment_info(self, frame, result: AssessmentResult):
            """Draw assessment information on frame"""
            if not result:
                return frame
            
            h, w = frame.shape[:2]
            
            # Background panel for assessment info
            panel_height = 150
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, h - panel_height - 10), (500, h - 10), (0, 0, 0), -1)
            frame = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)
            
            # Assessment text
            y_offset = h - panel_height + 20
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.6
            thickness = 2
            
            # Muscle group and side
            text = f"{self.muscle_group} ({self.side})"
            cv2.putText(frame, text, (20, y_offset), font, font_scale, (255, 255, 255), thickness)
            y_offset += 25
            
            # Display label
            text = f"Result: {result.display_label}"
            cv2.putText(frame, text, (20, y_offset), font, font_scale, (255, 255, 255), thickness)
            y_offset += 25
            
            # Pain score
            color = (0, 255, 0) if result.pain_score <= 3 else (0, 165, 255) if result.pain_score <= 7 else (0, 0, 255)
            text = f"Pain Score: {result.pain_score}/10 ({result.categorical_pain_level})"
            cv2.putText(frame, text, (20, y_offset), font, font_scale, color, thickness)
            y_offset += 25
            
            # Angle or displacement if available
            if 'angle' in result.additional_data:
                text = f"Angle: {result.additional_data['angle']:.1f}°"
                cv2.putText(frame, text, (20, y_offset), font, font_scale, (255, 255, 255), thickness)
            elif 'absNormalizedDisplacement' in result.additional_data:
                text = f"Displacement: {result.additional_data['absNormalizedDisplacement']:.3f}"
                cv2.putText(frame, text, (20, y_offset), font, font_scale, (255, 255, 255), thickness)
            
            return frame
        
        def _should_process_frame(self) -> bool:
            """
            Check if frame should be processed (frame throttling).
            Matches Flutter's throttling logic: minimum 120ms between processed frames.
            """
            if self.processing_frame:
                return False
            
            now_ms = int(time.time() * 1000)
            if self.last_processed_time is None:
                self.last_processed_time = now_ms
                return True
            
            elapsed_ms = now_ms - self.last_processed_time
            if elapsed_ms < self.min_frame_interval_ms:
                return False
            
            self.last_processed_time = now_ms
            return True
        
        def _process_frame(self, frame):
            """
            Process a single frame for pose detection and assessment.
            This replicates Flutter's _startImageStream() callback logic exactly.
            """
            if not self._should_process_frame():
                return
            
            self.processing_frame = True
            try:
                # Step 1: Detect poses (matching Flutter's _poseService.detectFromCameraImage)
                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = self.detector.pose.process(image_rgb)
                
                if results.pose_landmarks:
                    try:
                        # Step 2: Extract landmarks (matching Flutter's _poseService.getPoseLandmarks)
                        landmarks = self.detector.detect_landmarks_from_frame(frame)
                        
                        # Step 3: Validate landmarks (matching Flutter's validation)
                        if landmarks is None or len(landmarks) == 0:
                            # Flutter: debugPrint('Warning: Empty landmarks detected, skipping frame')
                            return
                        
                        if len(landmarks) < 5:
                            # Flutter: debugPrint('Warning: Insufficient landmarks detected, skipping frame')
                            return
                        
                        # Step 4: Run assessment (matching Flutter's AssessmentService.assess)
                        try:
                            self.current_result = AssessmentService.assess(
                                self.muscle_group, landmarks, self.side
                            )
                        except Exception as e:
                            # Flutter: debugPrint('Assessment failed: $e')
                            pass
                    
                    except Exception as e:
                        # Flutter: debugPrint('Landmark processing error: $e')
                        pass
                
            except Exception as e:
                # Flutter: debugPrint('Pose detection error: $e')
                pass
            finally:
                self.processing_frame = False
        
        def run(self):
            """
            Run real-time assessment loop.
            Replicates Flutter's camera stream processing flow.
            """
            if not self.cap:
                self.start_camera()
            
            print(f"\nStarting real-time assessment for {self.muscle_group} ({self.side})")
            print("Press 'q' to quit, 's' to toggle skeleton, 'a' to toggle assessment info")
            print("Press '1-9' to change muscle group, 'l/r' to change side\n")
            
            try:
                while True:
                    ret, frame = self.cap.read()
                    if not ret:
                        print("Failed to read frame")
                        break
                    
                    # Flip frame horizontally for mirror effect (matching Flutter's front camera behavior)
                    frame = cv2.flip(frame, 1)
                    
                    # Process frame for pose detection and assessment (matching Flutter's flow)
                    self._process_frame(frame)
                    
                    # Draw skeleton if enabled (for visualization only)
                    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    results = self.detector.pose.process(image_rgb)
                    if self.show_skeleton and results.pose_landmarks:
                        self.detector.draw_landmarks(frame, results)
                    
                    # Draw assessment info if enabled
                    if self.show_assessment and self.current_result:
                        frame = self.draw_assessment_info(frame, self.current_result)
                    
                    # Calculate and display FPS (matching Flutter's performance monitoring)
                    self.frame_count += 1
                    now_ms = int(time.time() * 1000)
                    if self.last_fps_time == 0:
                        self.last_fps_time = now_ms
                    elif now_ms - self.last_fps_time >= 1000:
                        self.current_fps = self.frame_count * 1000.0 / (now_ms - self.last_fps_time)
                        self.frame_count = 0
                        self.last_fps_time = now_ms
                        
                        # Log performance metrics (matching Flutter's warning)
                        if self.current_fps < 8.0:
                            print(f"Warning: Low FPS detected: {self.current_fps:.1f}")
                    
                    # Display FPS on frame
                    cv2.putText(frame, f"FPS: {self.current_fps:.1f}", (10, 30), 
                              cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    
                    # Display frame
                    cv2.imshow('AROM Assessment - Real-Time', frame)
                    
                    # Handle keyboard input
                    key = cv2.waitKey(1) & 0xFF
                    if key == ord('q'):
                        break
                    elif key == ord('s'):
                        self.show_skeleton = not self.show_skeleton
                        print(f"Skeleton display: {'ON' if self.show_skeleton else 'OFF'}")
                    elif key == ord('a'):
                        self.show_assessment = not self.show_assessment
                        print(f"Assessment info: {'ON' if self.show_assessment else 'OFF'}")
                    elif key == ord('l'):
                        self.side = 'Left'
                        print(f"Side changed to: {self.side}")
                    elif key == ord('r'):
                        self.side = 'Right'
                        print(f"Side changed to: {self.side}")
                    elif key >= ord('1') and key <= ord('9'):
                        muscle_groups = ['Triceps', 'Biceps', 'Shoulders', 'Chest', 
                                        'Quadriceps', 'Hamstrings', 'Gluteals', 'Calves', 'Abdominals']
                        idx = key - ord('1')
                        if idx < len(muscle_groups):
                            self.muscle_group = muscle_groups[idx]
                            print(f"Muscle group changed to: {self.muscle_group}")
            
            except KeyboardInterrupt:
                print("\nStopped by user")
            finally:
                self.cleanup()
        
        def cleanup(self):
            """Clean up resources"""
            if self.cap:
                self.cap.release()
            cv2.destroyAllWindows()
            print("Camera released")
    
    print("RealTimeAROMAssessment class loaded successfully")
else:
    print("Real-time assessment not available - install MediaPipe and OpenCV first")


### Usage: Start Real-Time Assessment

Uncomment and run the cell below to start real-time assessment from your webcam.


In [ ]:
# Uncomment to start real-time assessment:
# if MEDIAPIPE_AVAILABLE:
#     # Create assessment instance
#     # Parameters:
#     #   muscle_group: 'Triceps', 'Biceps', 'Shoulders', 'Chest', 'Quadriceps', 
#     #                 'Hamstrings', 'Gluteals', 'Calves', 'Abdominals', etc.
#     #   side: 'Left' or 'Right' (ignored for trunk assessments)
#     #   camera_index: 0 for default camera, 1 for second camera, etc.
#     #   show_skeleton: True to show pose skeleton overlay
#     #   show_assessment: True to show assessment results
#     
#     assessor = RealTimeAROMAssessment(
#         muscle_group='Triceps',
#         side='Right',
#         camera_index=0,
#         show_skeleton=True,
#         show_assessment=True
#     )
#     
#     # Start real-time assessment
#     assessor.run()
# else:
#     print("Please install MediaPipe and OpenCV first: pip install mediapipe opencv-python")

print("Real-time assessment ready. Uncomment the code above to start.")


### Alternative: Process Single Image

You can also process a single image file instead of using the camera.


### Implementation Comparison: Flutter vs Notebook

**Flutter Implementation (`c_camera.dart` + `pose_detection_service.dart`):**
1. Uses Google ML Kit Pose Detection (mobile-only)
2. `_startImageStream()` starts camera image stream
3. For each frame:
   - Throttles processing (120ms minimum interval)
   - `_poseService.detectFromCameraImage()` → returns `List<Pose>`
   - `_poseService.getPoseLandmarks()` → normalizes to **0-1 coordinates**
   - Validates landmarks (empty check, minimum 5 landmarks)
   - `AssessmentService.assess()` → runs assessment
   - Updates state with results

**Notebook Implementation:**
1. Uses MediaPipe Pose (cross-platform Python alternative)
2. `RealTimeAROMAssessment.run()` starts camera loop
3. For each frame:
   - `_should_process_frame()` → throttles (120ms minimum interval) ✓
   - `_process_frame()` → detects poses using MediaPipe ✓
   - `detect_landmarks_from_frame()` → normalizes to **0-1 coordinates** ✓
   - Validates landmarks (empty check, minimum 5 landmarks) ✓
   - `AssessmentService.assess()` → runs assessment ✓
   - Updates results ✓

**Key Matching Points:**
- ✅ Same coordinate system: Normalized 0-1 range (not pixels)
- ✅ Same frame throttling: 120ms minimum between processed frames
- ✅ Same landmark validation: Empty check + minimum 5 landmarks
- ✅ Same assessment flow: Pose detection → Landmark extraction → Assessment
- ✅ Same error handling: Try-catch blocks matching Flutter's debugPrint statements
- ✅ Same performance monitoring: FPS tracking with low FPS warnings

**Differences (Platform-Specific):**
- Flutter: Google ML Kit (mobile-only, optimized for Android/iOS)
- Notebook: MediaPipe (cross-platform, works on desktop/laptop)
- Both provide 33 body landmarks with normalized coordinates (0-1)


### Keyboard Controls (Real-Time Mode)

When running real-time assessment:
- **'q'** - Quit and close camera
- **'s'** - Toggle skeleton overlay on/off
- **'a'** - Toggle assessment info display on/off
- **'l'** - Switch to Left side
- **'r'** - Switch to Right side
- **'1'** - Switch to Triceps
- **'2'** - Switch to Biceps
- **'3'** - Switch to Shoulders
- **'4'** - Switch to Chest
- **'5'** - Switch to Quadriceps
- **'6'** - Switch to Hamstrings
- **'7'** - Switch to Gluteals
- **'8'** - Switch to Calves
- **'9'** - Switch to Abdominals


## Comparison: ML Kit vs MediaPipe

### Google ML Kit (Used in Flutter App)
- **Platform**: Android/iOS only (via Flutter)
- **Package**: `google_mlkit_pose_detection`
- **Landmarks**: 33 body landmarks
- **Format**: Returns `Pose` objects with normalized coordinates
- **Real-time**: Optimized for mobile devices

### MediaPipe (Python Alternative)
- **Platform**: Python, Jupyter, Web, Mobile
- **Package**: `mediapipe`
- **Landmarks**: 33 body landmarks (same as ML Kit)
- **Format**: Returns normalized coordinates (0-1 range)
- **Real-time**: Works well on desktop/laptop

### Key Differences
1. **Coordinate System**: Both use normalized coordinates, but conversion may be needed
2. **Landmark Names**: MediaPipe uses different naming (e.g., `LEFT_SHOULDER` vs `leftShoulder`)
3. **Confidence Scores**: MediaPipe provides visibility/confidence scores per landmark
4. **Performance**: ML Kit is optimized for mobile; MediaPipe works cross-platform

### Recommendation
For testing and validation in Jupyter, MediaPipe is the best alternative since it:
- Provides the same 33 landmarks as ML Kit
- Works seamlessly in Python/Jupyter
- Can process images and video
- Has good documentation and examples
